## Резюме

Вот что мы рассмотрели в этой лекции.
- Сначала мы представили проблему классификации и кластеризации для векторных данных и важность выбора правильной меры для вычисления (нес)сходств между образцами.
- Затем мы представили три подхода к вычислению (нес)сходств в многомерных временных рядах.

1. DTW, метрика на основе выравнивания.

2. TCK, ядерное сходство.

3. Встраивание RC, подход к встраиванию временных рядов в векторные данные.
- Эти меры (нес)сходства являются краеугольным камнем в классификации и кластеризации временных рядов.
- После вычисления мы увидели, как их можно легко подключить к тому же методу классификации и кластеризации, который мы видели для векторных данных.

В заключение мы выделим основные плюсы и минусы трех подходов к вычислению (не)сходства MTS.

**DTW**

- ✅ В большинстве случаев хорошо работает с гиперпараметрами по умолчанию.
- ✅ Инвариантен к перемещениям во времени.
- ❌ Медленно.
- ❌ Не учитывает сложные динамические особенности.

**TCK**

- ✅ Гиперпараметры легко настраиваются.
- ✅ Обрабатывает отсутствующие данные.
- ❌ Очень медленно.

**RC-embedding**

- ✅ Быстро.
- ✅ Захватывает сложные динамические особенности.
- ❌ Множество гиперпараметров для установки.

- Каждый подход может достигать лучшей или худшей производительности в зависимости от данных и решаемой проблемы.
- Выбор оптимальной меры (несходства), алгоритма классификации/кластеризации и гиперпараметров часто является сложной процедурой.
- Это требует опыта и должно выполняться с использованием систематических подходов, таких как перекрестная проверка.

---

## Упражнения

Мы рассмотрим набор данных [Libras](https://www.timeseriesclassification.com/description.php?Dataset=Libras). Набор данных содержит 15 классов по 24 экземпляра в каждом. Каждый класс ссылается на тип движения руки в LIBRAS (португальское название «LÍngua BRAsileira de Sinais», официальный бразильский язык сигналов).

### Упражнение 1

1. Вычислите матрицу расстояний DTW.
2. Получите из нее матрицу сходства.
3. Постройте две матрицы. Прокомментируйте структуру, которую вы видите (не забудьте отсортировать элементы по классам).
4. Выполните классификацию с помощью классификаторов SVC и $k$-NN и сообщите:
- время обучения и тестирования,
- точность и оценку F1 на тестовом наборе.
5. Выполните иерархическую кластеризацию с помощью алгоритма Linkage Ward.
6. Постройте дендрограмму и проверьте ее, чтобы выбрать оптимальный порог для генерации кластерного раздела. Сообщите NMI для найденного вами раздела.
7. Выполните снижение размерности с помощью KernelPCA. Постройте результаты на двумерном графике.

In [2]:
!pip install dtaidistance tck reservoir_computing tsa_course

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 19.9 MB/s eta 0:00:00


In [7]:
import time
from tck.datasets import DataLoader
from dtaidistance import dtw_ndim
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, v_measure_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.decomposition import KernelPCA
import scipy.spatial.distance as ssd

In [8]:
Xtr, Ytr, Xte, Yte = DataLoader().get_data('Libras')

Loaded Libras dataset.
Number of classes: 15
Data shapes:
  Xtr: (180, 45, 2)
  Ytr: (180, 1)
  Xte: (180, 45, 2)
  Yte: (180, 1)


In [ ]:
dist_matrix_dtw = dtw_ndim.distance_matrix(Xtr)

In [5]:
gamma = 1.0 / (dist_matrix_dtw.mean()**2)
sim_matrix_dtw = np.exp(-gamma * (dist_matrix_dtw**2))

NameError: name 'dist_matrix_dtw' is not defined

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(dist_matrix_dtw, cmap='viridis')
ax[0].set_title("DTW Distance Matrix")
ax[1].imshow(sim_matrix_dtw, cmap='hot')
ax[1].set_title("DTW Similarity Matrix")
plt.show()

In [ ]:
start = time.time()
knn = KNeighborsClassifier(n_neighbors=1, metric='precomputed')
knn.fit(dist_matrix_dtw, Ytr.ravel())
train_preds = knn.predict(dist_matrix_dtw)
end = time.time()

In [ ]:
print(f"KNN (DTW) Accuracy: {accuracy_score(Ytr, train_preds):.4f}")
print(f"Time: {end - start:.2f}s")

In [ ]:
dist_array = ssd.squareform(dist_matrix_dtw)
Z = linkage(dist_array, method='ward')

plt.figure(figsize=(10, 5))
dendrogram(Z)
plt.title("Dendrogram (DTW + Ward)")
plt.show()
clusters = fcluster(Z, t=15, criterion='maxclust')
print(f"Clustering V-measure: {v_measure_score(Ytr.ravel(), clusters):.4f}")

In [ ]:
kpca = KernelPCA(n_components=2, kernel='precomputed')
X_kpca = kpca.fit_transform(sim_matrix_dtw)

plt.figure(figsize=(8, 6))
plt.scatter(X_kpca[:, 0], X_kpca[:, 1], c=Ytr.ravel(), cmap='tab20')
plt.title("KernelPCA (DTW Similarity)")
plt.colorbar(label='Class')
plt.show()

### Упражнение 2

1. Добавьте 40% пропущенных значений к обучающим и тестовым данным.
2. Вычислите ядро ​​TCK.
3. Вычислите матрицу различий из ядра (попробуйте сделать дополнительное к тому, что вы сделали для получения сходства из DTW).

Повторите пункты 4–7 из предыдущего упражнения.

### Упражнение 3

1. Вычислите RC-вложения.
2. Получите матрицу сходства и различия из представлений MTS.

Повторите пункты 4–7 из предыдущих упражнений.